# Deep Network Development, 2026 Autumn - Week 8 Practice - **Exercise sheet**

## Exercises

---

### Exercise 8.1: Binary classification with Dataset and DataLoader

Predict a penguin's recorded sex (female: 0, male: 1) from bill length,
bill depth, and flipper length. Handle missing input values, create a
PyTorch dataset and loaders, then train and test a binary classifier.

#### Provided setup

Download the original [Palmer Penguins CSV](https://allisonhorst.github.io/palmerpenguins/).
Keep rows with known sex. These 333 rows have complete measurements. To practise imputation, the setup
deliberately hides 20 bill lengths using a fixed random seed.

Shuffle inputs and labels together, then split 70%/15%/remaining into training,
validation, and test sets. Run this code unchanged.

In [ ]:
import csv
import io
import urllib.request
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.set_num_threads(1)
DATA_URL = (
    'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/'
    '156daa4301838d9fdcc5b018b29b9149bf975552/inst/extdata/penguins.csv'
)
with urllib.request.urlopen(DATA_URL, timeout=30) as stream:
    records = list(csv.DictReader(io.StringIO(stream.read().decode('utf-8'))))

FEATURE_NAMES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
LABEL_NAMES = ['sex']
sex_dict = {'female': 0, 'male': 1}
usable = [row for row in records if row['sex'] in sex_dict]
inputs = np.array([
    [float('nan') if row[name] == 'NA' else float(row[name]) for name in FEATURE_NAMES]
    for row in usable
], dtype=np.float32)
labels = np.array([sex_dict[row['sex']] for row in usable], dtype=np.float32)
assert not np.isnan(inputs).any()

hidden_rows = np.random.default_rng(0).choice(len(inputs), size=20, replace=False)
inputs[hidden_rows, 0] = np.nan

order = np.random.default_rng(42).permutation(len(inputs))
inputs, labels = inputs[order], labels[order]
TRAINING_SET_RATIO, VALIDATION_SET_RATIO = 0.7, 0.15
validation_offset = int(len(inputs) * TRAINING_SET_RATIO)
test_offset = validation_offset + int(len(inputs) * VALIDATION_SET_RATIO)
inputs_train_raw, labels_train = inputs[:validation_offset], labels[:validation_offset]
inputs_val_raw, labels_val = inputs[validation_offset:test_offset], labels[validation_offset:test_offset]
inputs_test_raw, labels_test = inputs[test_offset:], labels[test_offset:]

print('Feature columns:', FEATURE_NAMES)
print('Label columns:', LABEL_NAMES, '(female=0, male=1)')

for name, xs, ys in [('Training', inputs_train_raw, labels_train),
                     ('Validation', inputs_val_raw, labels_val),
                     ('Test', inputs_test_raw, labels_test)]:
    print(f'{name}: inputs {xs.shape}, labels {ys.shape}, missing bill lengths {np.isnan(xs[:, 0]).sum()}')
# Expected sizes: 233 training, 49 validation, 51 test.


#### Task 1: Mean imputation and a missingness flag

Implement `prepare_inputs(input_split, fill_value)`: mask missing bill lengths
with `np.isnan`, replace them in a copy, and append the mask as a `float32`
column. The flag is 1 for a missing value and 0 otherwise.

Use the supplied training-set mean for all splits. Keep observed values and
the original array unchanged. The output shape should be `(N, 4)`.


In [ ]:
bill_length_mean = np.nanmean(inputs_train_raw[:, 0])

def prepare_inputs(input_split, fill_value):
    # TODO: mask missing bill lengths, replace them in a copy, and append the flag.
    # SOLUTION 1 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 1 END

inputs_train = prepare_inputs(inputs_train_raw, bill_length_mean)
inputs_val = prepare_inputs(inputs_val_raw, bill_length_mean)
inputs_test = prepare_inputs(inputs_test_raw, bill_length_mean)

# Check replacement, flag values, and preservation of the original array.
toy_inputs = np.array([[10., 2., 30.], [np.nan, 4., 40.]], dtype=np.float32)
expected = np.array([[10., 2., 30., 0.], [15., 4., 40., 1.]], dtype=np.float32)
assert np.allclose(prepare_inputs(toy_inputs, 15.), expected)
assert np.isnan(toy_inputs[1, 0])
for raw, filled in [(inputs_train_raw, inputs_train),
                    (inputs_val_raw, inputs_val),
                    (inputs_test_raw, inputs_test)]:
    mask = np.isnan(raw[:, 0])
    assert filled.shape == (len(raw), 4) and np.isfinite(filled).all()
    assert np.array_equal(filled[:, 3], mask.astype(np.float32))
    assert np.allclose(filled[mask, 0], bill_length_mean)
    assert np.allclose(filled[~mask, :3], raw[~mask])
print('Task 1 passed! Training mean:', bill_length_mean)

print('Feature columns after preprocessing:', FEATURE_NAMES + ['bill_length_missing'])


In [ ]:
# Provided: standardize the three measurements
feature_means = inputs_train[:, :3].mean(axis=0)
feature_stds = inputs_train[:, :3].std(axis=0)
# Reuse training statistics, keep the 0/1 flag unchanged
for split_inputs in (inputs_train, inputs_val, inputs_test):
    split_inputs[:, :3] = (split_inputs[:, :3] - feature_means) / feature_stds


#### Task 2: Dataset and DataLoader

Implement `BinaryClassificationDataset`. Store inputs and labels as `torch.float32`
tensors. `__len__` returns the number of examples. `__getitem__(idx)` returns
one input vector of shape `(4,)` and one label of shape `(1,)`.

Create a dataset and a `DataLoader` for each split, with batch size 32.
Enable shuffling for the training and validation loaders. Batches should have shapes
`(B, 4)` and `(B, 1)`, including the last smaller batch.


In [ ]:
class BinaryClassificationDataset(Dataset):
    def __init__(self, input_split, labels_split):
        # TODO: store inputs in self.xs and labels in self.ys as float32 tensors.
        # SOLUTION 2 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 2 END

    def __len__(self):
        # TODO: return the number of examples.
        # SOLUTION 3 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 3 END

    def __getitem__(self, idx):
        # TODO: return one input vector and one label with shape (1,).
        # SOLUTION 4 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 4 END

train_dataset = BinaryClassificationDataset(inputs_train, labels_train)
val_dataset = BinaryClassificationDataset(inputs_val, labels_val)
test_dataset = BinaryClassificationDataset(inputs_test, labels_test)

assert len(train_dataset) == len(inputs_train)
sample_x, sample_y = train_dataset[0]
assert sample_x.shape == (4,) and sample_y.shape == (1,)
assert sample_x.dtype == sample_y.dtype == torch.float32
assert torch.equal(sample_y, torch.tensor([labels_train[0]]))
print('One example:', sample_x.shape, sample_y.shape)

BATCH_SIZE = 32

# TODO: create the 3 DataLoaders, only the training and validation loaders should shuffle.
# SOLUTION 5 START
raise NotImplementedError("Complete this part")
# SOLUTION 5 END

for split_name, loader in [('Training', dataloader_train),
                           ('Validation', dataloader_val),
                           ('Test', dataloader_test)]:
    batch_xs, batch_ys = next(iter(loader))
    assert batch_xs.shape == (BATCH_SIZE, 4) and batch_ys.shape == (BATCH_SIZE, 1)
    print(f'{split_name} batch shapes:', batch_xs.shape, batch_ys.shape)
print('Task 2 passed!')


#### Task 3: Binary classification network

Implement `BinaryNet` using `nn.Sequential`, with the appropriate input and output dimension, and 2 hidden layers with dimensions 30 and 20 respectively. A ReLU should follow both hidden layers.
Store the sequence in `self.network`, `forward` should apply it to the input.
Instantiate the network as `binary_model`.


In [ ]:
torch.manual_seed(7)

class BinaryNet(nn.Module):
    def __init__(self, n_input_vars):
        # TODO: initialize nn.Module and define self.network using nn.Sequential.
        # SOLUTION 6 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 6 END

    def forward(self, x):
        # TODO: make predictions by applying self.network to x.
        # SOLUTION 7 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 7 END

# TODO: create the model with the correct number of input variables.
# SOLUTION 8 START
raise NotImplementedError("Complete this part")
# SOLUTION 8 END

with torch.no_grad():
    example_probabilities = binary_model(batch_xs)
assert isinstance(binary_model.network, nn.Sequential)
assert isinstance(binary_model.network[-1], nn.Sigmoid)
assert example_probabilities.shape == batch_ys.shape
assert torch.all((example_probabilities >= 0) & (example_probabilities <= 1))
print('Model output shape:', example_probabilities.shape)


#### Task 4: Binary accuracy

Implement `bin_acc_fn(y_hat, y_true)` for tensors shaped `(B, 1)`.
Convert probabilities to labels with `torch.round`, compare them with the targets,
and return the fraction of correct predictions as a scalar tensor.


In [ ]:
@torch.no_grad()
def bin_acc_fn(y_hat, y_true):
    assert y_hat.shape == y_true.shape
    # TODO: convert probabilities to labels and calculate the fraction correct.
    # SOLUTION 9 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 9 END

toy_probabilities = torch.tensor([[0.2], [0.7], [0.8], [0.4]])
toy_targets = torch.tensor([[0.], [1.], [0.], [0.]])
assert torch.isclose(bin_acc_fn(toy_probabilities, toy_targets), torch.tensor(0.75))
assert bin_acc_fn(toy_probabilities, toy_targets).ndim == 0
print('Task 4 passed!')


#### Task 5: Training and validation

Complete the marked steps. Use the appropriate loss and Adam optimizer with learning rate
0.001. Train for at most 50 epochs and calculate training/validation loss and accuracy.

Use early stopping with patience 5 based on validation loss, and restore the best model weights.

Use the provided plots to inspect learning.


In [ ]:
n_epochs = 50
learning_rate = 0.001
# TODO: choose the loss for a sigmoid probability and create an Adam optimizer.
# SOLUTION 10 START
raise NotImplementedError("Complete this part")
# SOLUTION 10 END


In [ ]:
from copy import deepcopy

patience = 5
best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_state = deepcopy(binary_model.state_dict())

train_losses, val_losses = [], []
train_accuracy, val_accuracy = [], []

for epoch_idx in range(n_epochs):
    # Training
    binary_model.train()
    train_losses_in_epoch, train_accs_in_epoch = [], []

    for train_batch_xs, train_batch_ys in dataloader_train:
        # TODO: clear gradients from the previous batch.
        # SOLUTION 11 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 11 END
        # TODO: make predictions for this batch.
        # SOLUTION 12 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 12 END
        # TODO: calculate the scalar loss and binary accuracy.
        # SOLUTION 13 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 13 END
        # TODO: save loss.detach() and accuracy in their epoch lists.
        # SOLUTION 14 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 14 END
        # TODO: backpropagate the loss.
        # SOLUTION 15 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 15 END
        # TODO: update the model parameters.
        # SOLUTION 16 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 16 END

    # Average batch means
    avg_tr_epoch_loss = torch.stack(train_losses_in_epoch).mean()
    avg_tr_epoch_acc = torch.stack(train_accs_in_epoch).mean()
    train_losses.append(avg_tr_epoch_loss.item())
    train_accuracy.append(avg_tr_epoch_acc.item())

    # Validation
    binary_model.eval()
    val_losses_in_epoch, val_accs_in_epoch = [], []
    for val_batch_xs, val_batch_ys in dataloader_val:
        with torch.no_grad():
            # TODO: make validation predictions, then calculate loss and accuracy.
            # SOLUTION 17 START
            raise NotImplementedError("Complete this part")
            # SOLUTION 17 END
        val_losses_in_epoch.append(loss)
        val_accs_in_epoch.append(acc)

    avg_val_epoch_loss = torch.stack(val_losses_in_epoch).mean()
    avg_val_epoch_acc = torch.stack(val_accs_in_epoch).mean()
    val_losses.append(avg_val_epoch_loss.item())
    val_accuracy.append(avg_val_epoch_acc.item())

    if (epoch_idx + 1) % 10 == 0:
        print(f'Epoch #{epoch_idx + 1}, tr loss: {avg_tr_epoch_loss:.4f}, '
              f'val loss: {avg_val_epoch_loss:.4f}, tr acc: {avg_tr_epoch_acc:.3f}, '
              f'val acc: {avg_val_epoch_acc:.3f}')

    # TODO: track the best validation loss and weights, stop after patience epochs without improvement.
    # SOLUTION 21 START
    raise NotImplementedError("Complete this part")
    # SOLUTION 21 END

# Restore the best validation model before testing.
binary_model.load_state_dict(best_model_state)

assert 0 < len(train_losses) == len(val_losses) <= n_epochs
assert np.isfinite(train_losses).all() and np.isfinite(val_losses).all()
assert len(train_accuracy) == len(val_accuracy) == len(train_losses)
assert all(not loss.requires_grad for loss in train_losses_in_epoch)
print('Training finished.')


In [ ]:
# Supplied plotting code
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(train_losses, label='Training')
axes[0].plot(val_losses, label='Validation')
axes[0].set(xlabel='Epochs', ylabel='BCE loss', title='Loss', ylim=(0, None))
axes[1].plot(train_accuracy, label='Training')
axes[1].plot(val_accuracy, label='Validation')
axes[1].set(xlabel='Epochs', ylabel='Binary accuracy', title='Accuracy', ylim=(0, 1.05))
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


#### Task 6: Test the trained model

Evaluate batches from `dataloader_test` without gradients,
collect predictions and their matching labels. Calculate BCE loss and binary accuracy over the entire
test set.


In [ ]:
binary_model.eval()
test_ys_preds, test_ys = [], []

for test_batch_xs, test_batch_ys in dataloader_test:
    with torch.no_grad():
        # TODO: make test predictions and collect predictions and matching labels.
        # SOLUTION 18 START
        raise NotImplementedError("Complete this part")
        # SOLUTION 18 END

# TODO: concatenate all batches along the examples dimension.
# SOLUTION 19 START
raise NotImplementedError("Complete this part")
# SOLUTION 19 END

# TODO: calculate test loss and accuracy using all test predictions and labels.
# SOLUTION 20 START
raise NotImplementedError("Complete this part")
# SOLUTION 20 END

assert test_ys_preds.shape == test_ys.shape == (len(test_dataset), 1)
assert not test_ys_preds.requires_grad
assert torch.equal(test_ys[:, 0], test_dataset.ys)
print(f'Test loss: {test_loss:.4f}, test accuracy: {test_acc:.3f}')

print('First five P(male):', test_ys_preds[:5, 0].tolist())
print('Predicted labels:', torch.round(test_ys_preds[:5, 0]).tolist())
print('Actual labels:   ', test_ys[:5, 0].tolist())
